# Nettoyage des données

Nous comparons les traitements décrits dans `Amaury_nettoyage.py`, `nettoyage_franck.py`, `nettoyage_Taima.py` et `bdr_nettoye.py`. La comparaison repose sur la lecture du code. Les résultats sur les fichiers de données restent à vérifier par l’exécution.

## 1. Ce que les scripts ont en commun

| Point commun | Scripts concernés | Ce que fait le code |
| --- | --- | --- |
| Lecture des données DVF et suppression des doublons | Amaury, Franck, Taima et BDR | Les quatre utilisent pandas pour lire un CSV DVF et suppriment les lignes identiques au moment du dédoublonnage. |
| Sélection des colonnes et des types de biens | Amaury et Franck | Ils conservent les mêmes 12 colonnes DVF et sélectionnent les maisons et appartements. |
| Conversion des dates et des mesures | Amaury, Franck et BDR | Ils convertissent la date de mutation, la valeur foncière, la surface bâtie, le nombre de pièces et la surface du terrain. |
| Nettoyage du texte et des en-têtes | Amaury et BDR | Ils retirent les espaces en bordure des chaînes et suppriment les lignes d’en-tête répétées. |
| Indicateurs de loyers | Franck et Taima | Leurs fonctions lisent un CSV en `cp1252` séparé par des points-virgules, convertissent plusieurs colonnes numériques et suppriment les doublons. |

## 2. Où les scripts diffèrent

Les lignes sur le nettoyage concernent les données DVF. La dernière ligne décrit ce que chaque script lance lorsqu’on l’exécute directement.

| Critère | Amaury | Franck | Taima | BDR |
| --- | --- | --- | --- | --- |
| Colonnes DVF | Sélectionne 12 colonnes à la lecture. | Sélectionne les mêmes 12 colonnes après la lecture. | Conserve les colonnes lues. | Supprime une liste de colonnes. Si `type_local` et `code_type_local` sont présents, vérifie la cohérence des codes pour les maisons et appartements avant de retirer `code_type_local`. |
| Filtre des biens | Garde `Maison` et `Appartement` après avoir retiré les espaces autour des libellés. | Garde les libellés exacts `Maison` et `Appartement`. | Aucun filtre. | Exclut les libellés contenant `Dépendance` ou `Local industriel. commercial ou assimilé`, sans distinction de casse. |
| Codes géographiques | Lit les codes comme du texte et affiche des contrôles. | Convertit les codes postaux, communaux et départementaux en entiers, ce qui peut retirer les zéros initiaux. | Aucune conversion explicite de ces codes. | Lit le code postal comme du texte et retire un éventuel suffixe `.0`. |
| Valeurs manquantes | Conserve les champs numériques manquants et affiche des compteurs d’anomalies. | Remplace les valeurs manquantes de plusieurs colonnes numériques et codes par `0` avant conversion. | Aucun remplacement explicite. | Remplit les valeurs manquantes par `Non renseigné` ou `0` dans les colonnes prévues. |
| Dates invalides | Les compte et les convertit en valeurs manquantes. | Les convertit en valeurs manquantes. | Aucune conversion explicite des dates. | Lève une erreur lorsqu’une date renseignée ne peut pas être convertie. |
| En-têtes répétés | Retire une ligne si ses 12 valeurs correspondent aux noms des colonnes, après retrait des espaces. | Aucune détection explicite. | Aucune détection explicite. | Retire une ligne si sa valeur `id_mutation` contient exactement le nom de cette colonne après retrait des espaces. |
| Moment du dédoublonnage | Après la lecture, puis après les conversions et le filtrage. | Après le filtrage et les conversions. | Dès la lecture du CSV. | Après le nettoyage du texte, avant le filtrage des biens et les conversions numériques. |
| Exécution et export | Traite les fichiers correspondant à `dvf *.csv` dans `Data/` et écrit un CSV sans index par fichier dans `Data cleaned/`. | Parcourt `../Data/Bronze/`, traite les fichiers dont le nom commence par `dvf` ou `indicateurs`, et écrit les CSV avec index dans `../Data/Silver/`. | Exporte les revenus de l’Excel vers `revenus_nettoyes.csv` sans index. Les appels DVF et loyers sont commentés. | Traite le fichier DVF des Bouches-du-Rhône défini dans le script et écrit `bdr_clean.csv` sans index. |

## 3. Règles proposées pour le traitement DVF commun

Les règles ci-dessous suivent l’ordre du traitement. La section suivante présente les choix à valider sur le CSV de référence.

| Étape du nettoyage | Règle proposée | Pourquoi ce choix |
| --- | --- | --- |
| 1. Lire les colonnes | Garder les 12 colonnes communes à Amaury et Franck. Lire les codes géographiques comme du texte. Si `code_type_local` est présent, le lire aussi pour vérifier sa cohérence avec le libellé avant de le retirer. | Les codes en texte gardent leurs zéros initiaux. BDR fournit le contrôle de cohérence pour les maisons et appartements. |
| 2. Nettoyer le texte et les en-têtes | Normaliser les caractères invisibles et retirer les espaces en bordure. Supprimer une ligne d’en-tête répétée si ses valeurs correspondent aux noms de toutes les colonnes conservées. | La normalisation vient de BDR. Le contrôle sur la ligne entière vient d’Amaury. |
| 3. Filtrer les biens et contrôler les codes | Garder les maisons et appartements après nettoyage des libellés. Compter les codes géographiques inhabituels et examiner des exemples avant de décider s’il faut corriger des valeurs comme un code postal terminé par `.0`. | Le filtre reprend celui d’Amaury et de Franck. BDR retire le suffixe `.0` des codes postaux. |
| 4. Convertir les dates et les mesures | Compter séparément les champs vides et illisibles, ainsi que les zéros pour les mesures. Laisser les champs manquants vides, conserver la précision des prix et des surfaces et signaler les nombres de pièces fractionnaires. | Les contrôles d’Amaury distinguent déjà plusieurs de ces cas. Franck et BDR imputent certains champs à zéro, ce qui efface la distinction avec un zéro déjà présent. |
| 5. Contrôler les doublons | Compter les doublons exacts après nettoyage du texte et vérifier séparément ceux qui apparaissent après conversion. Comparer la ligne entière. | Deux valeurs illisibles différentes peuvent devenir vides après conversion. Aucun des quatre scripts ne déduplique sur le seul `id_mutation`. |
| 6. Exporter et rendre compte | Écrire un CSV sans index et présenter dans le notebook les volumes avant et après, les retraits par étape et des exemples d’anomalies. | Amaury, Taima et BDR exportent sans index. Le bilan permettra de vérifier les effets de chaque règle. |

## 4. Choix de nettoyage à valider

Examiner le CSV DVF de référence, puis renseigner la dernière colonne. Les trois premiers choix déterminent quelles lignes restent dans le fichier nettoyé.

| Décision | Options et effet sur le résultat | Proposition de départ | Choix retenu |
| --- | --- | --- | --- |
| **1. Natures de mutation** | Garder toutes les natures pour les maisons et appartements, ou retenir certains libellés de `nature_mutation`. Ce choix définit le périmètre de l’analyse. | Lister les libellés présents et choisir selon l’objectif de l’étude. | À renseigner |
| **2. Champs indispensables** | Pour chaque champ, garder et signaler la ligne, ou l’écarter si la donnée indispensable est invalide. Examiner la date, la valeur foncière, l’identifiant et, lorsque le code est disponible, la cohérence du type de bien. | Garder les lignes et signaler les anomalies en attendant la validation des champs indispensables. Distinguer les champs vides, les valeurs illisibles, les zéros et les valeurs négatives. | À renseigner |
| **3. Doublons après conversion** | Garder et signaler les lignes devenues identiques seulement après conversion, ou conserver une seule occurrence. Deux valeurs illisibles différentes peuvent devenir vides. | Supprimer les doublons exacts après nettoyage du texte. Examiner séparément ceux apparus après conversion. | À renseigner |
| **4. Corrections automatiques** | Définir les corrections autorisées pour les codes, les arrondis des prix et des surfaces, et les nombres de pièces fractionnaires. Chaque correction peut modifier une valeur brute. | Garder les codes en texte et conserver la précision des prix et des surfaces. Examiner les exemples avant de retirer un suffixe postal `.0` ou de traiter les nombres de pièces fractionnaires. | À renseigner |
| **5. Fichier illisible ou colonne absente** | Arrêter le lot ou signaler le fichier en échec et poursuivre avec les autres. Définir les colonnes obligatoires. | Exiger les 12 colonnes communes, traiter `code_type_local` comme facultatif et récapituler les erreurs en fin de lot. | À renseigner |

## 5. Plan de mise en œuvre

Le CSV de référence sert d’abord à documenter les anomalies et à valider les choix de la section 4. Une fois ces choix renseignés, nous appliquerons les règles retenues et vérifierons le traitement sur un deuxième fichier. Les loyers et les revenus viendront ensuite.

| Étape | Travail proposé | Appui dans les scripts existants |
| --- | --- | --- |
| 1. Choisir un CSV DVF de référence | Relever le nombre de lignes, les colonnes et les valeurs manquantes avant le nettoyage. | Amaury affiche déjà plusieurs compteurs. |
| 2. Valider les règles communes | Examiner les cas du CSV de référence et remplir la colonne « Choix retenu » de la section 4. | Les différences entre scripts indiquent les règles à arbitrer. |
| 3. Construire le traitement DVF | Implémenter les règles validées dans le script Python et exporter un CSV sans index. | Reprendre le nettoyage du texte de BDR et les contrôles d’Amaury. |
| 4. Vérifier le résultat | Comparer les volumes avant et après, compter les lignes retirées à chaque étape et montrer des exemples d’anomalies. Vérifier le traitement sur un deuxième CSV. | Les scripts comptent déjà certaines anomalies. La vérification sur les données reste à faire. |
| 5. Ajouter les loyers et les revenus | Reprendre la fonction de revenus de Taima. Comparer les colonnes et conversions des fonctions de loyers de Franck et Taima avant de choisir les règles communes. | Les fonctions existent dans les scripts étudiés. |

Pour les loyers, Franck sélectionne des colonnes et Taima conserve les colonnes lues. Le choix commun dépendra des indicateurs nécessaires au projet. Pour les revenus, Taima est le seul des quatre scripts à proposer un traitement.

Le premier livrable sera un script Python qui lit un CSV DVF et produit un CSV nettoyé, accompagné d’un exemple commenté dans ce notebook.

## 6. Bilan de `Script_final_V2` par rapport à `Script_final`

La V2 approfondit la comparaison et propose une structure pour le futur script. Les deux notebooks contiennent des cellules Markdown. Ce bilan repose sur leur lecture et sur la vérification des chemins cités. Les traitements restent à tester sur les données.

### Points positifs

| Apport de la V2 | Ce qu’il apporte par rapport à `Script_final` |
| --- | --- |
| Comparaison plus précise | Détaille la lecture, les conversions champ par champ, l’ordre des opérations, les exports et la gestion des erreurs. Elle signale notamment qu’une valeur fractionnaire peut faire échouer une conversion directe en entier chez Amaury. |
| Lien avec les fichiers du projet | Relève des écarts concrets entre les scripts et les fichiers disponibles, comme le séparateur DVF attendu par Taima et le chemin `Data/Bronze/` utilisé par Franck. |
| Origine des règles | Indique pour chaque règle proposée le script dont elle s’inspire. Le contrôle de `code_type_local` et le traitement des doublons après conversion restent identifiables comme des décisions à valider. |
| Plan de réalisation | Décrit les fonctions envisagées, propose `dvf Paris.csv` comme fichier de référence, puis prévoit une vérification sur la Gironde et la Haute-Garonne. Détaille aussi les futurs traitements des loyers et des revenus. |

### Points négatifs et limites

| Écart ou limite | Conséquence pour le script final |
| --- | --- |
| Corrections proposées avant validation | La V2 propose d’arrondir les prix, les surfaces et les nombres de pièces, et de retirer le suffixe postal `.0`. Dans `Script_final`, la précision est conservée par défaut et les corrections restent à décider. L’arrondi évite une conversion directe en entier sur une valeur fractionnaire, mais il modifie la donnée. |
| Préférence différente pour les doublons | La V2 propose de supprimer les doublons apparus après conversion. `Script_final` propose d’abord de les examiner. Deux valeurs distinctes peuvent devenir identiques après un arrondi ou le remplacement d’une valeur illisible par un champ vide. |
| Deux choix de format sont moins visibles | La V2 demande de lire `code_type_local` et ne précise pas le comportement si cette colonne manque. Elle propose aussi de poursuivre après l’échec d’un fichier. `Script_final` présente le caractère facultatif de la colonne et la poursuite du lot comme des choix à valider. |
| Cas d’anomalies à compléter | Le tableau de décision de la V2 cite surtout la valeur foncière et la surface bâtie dans sa ligne sur les valeurs anormales. Il reste à décider explicitement si une date illisible ou un identifiant vide justifie le retrait d’une ligne. Pour un type de bien incohérent, il reste aussi possible de conserver et signaler la ligne. |
| Vérification BDR limitée à l’en-tête | Les mêmes 12 colonnes sont obtenues si l’on applique la liste de suppressions BDR à l’en-tête de 41 colonnes décrit dans la V2. Cela ne valide pas un export BDR réel. La copie locale sous `__pycache__` est ignorée par Git, et le fichier existe dans `origin/dev` sous `bouched_rohne/bdr_nettoye.py`. |
| Jointures à vérifier sur les données | Garder les codes en texte préserve leur forme. Il reste à vérifier que les codes commune du DVF, des loyers et des revenus correspondent effectivement avant de s’appuyer sur les jointures. |

Pour la suite, utiliser la V2 comme référence pour les détails d’implémentation et renseigner les choix de la section 4 sur le CSV de référence. Les propositions qui changent les valeurs ou le nombre de lignes doivent être validées avant d’être codées.

## 7. Choix du script de base

**Script retenu pour le DVF : `Amaury_nettoyage.py`.** Le choix s’appuie sur quatre critères observables dans le code : la prise en charge des CSV DVF de `Data/`, la conservation des codes et des valeurs manquantes, les contrôles d’anomalies et la gestion des exports par fichier.

| Script | Éléments qui motivent le choix |
| --- | --- |
| **Amaury** | Parcourt les CSV DVF de `Data/` à partir du chemin du script. Lit les 12 colonnes prévues en texte, filtre les maisons et appartements et compte plusieurs types d’anomalies. Écrit un CSV sans index par fichier via un fichier temporaire, et poursuit le lot si un fichier échoue. |
| Franck | Conserve les mêmes 12 colonnes et applique le même filtre sur les biens. Utilise un chemin relatif au dossier de lancement, convertit les codes géographiques en entiers et remplace plusieurs valeurs manquantes par zéro. Sa fonction sur les loyers sera utile lors de l’étape suivante. |
| Taima | Pour le DVF, lit le CSV et retire les doublons. L’appel DVF est commenté et le séparateur attendu diffère de celui des CSV DVF de `Data/`. Sa fonction de revenus est la seule parmi les quatre scripts. |
| BDR | Organise le nettoyage en fonctions courtes et apporte le contrôle de `code_type_local` ainsi que la normalisation des caractères invisibles. Son exécution directe vise un seul CSV et le traitement remplace certaines valeurs manquantes par zéro ou `Non renseigné`. |

Le script d’Amaury sert de point de départ. Reprendre de BDR la normalisation du texte et le contrôle facultatif de `code_type_local`, avec un traitement des incohérences à valider. Dans le code d’Amaury, revoir l’arrondi de la valeur foncière, les conversions directes en entier des surfaces bâties et du nombre de pièces, ainsi que la suppression des doublons après conversion. Ces règles dépendent des choix de la section 4.

Ce choix repose sur la lecture du code. Le comportement du futur script reste à vérifier sur le CSV de référence.